# Notebook 10 — Clustering: k-Means and Hierarchical

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 10 of 15  
**Time estimate:** 75 minutes

> Patient stratification, cell type discovery, gene module identification —
> unsupervised clustering is the entry point to almost every exploratory
> genomics analysis.

---
## Step 1 — Motivation

We often don't know the labels in advance. Which patients form clinically
distinct subgroups? Which genes co-regulate together? Clustering discovers
structure in unlabeled data. k-means and hierarchical clustering are the
two workhorses.

---
## Step 2 — Intuition

**k-means (Lloyd's algorithm):**
1. Initialize $k$ centroids (randomly or k-means++)
2. Assign each point to the nearest centroid
3. Update centroids to the mean of assigned points
4. Repeat until convergence

Minimizes within-cluster sum of squared distances (WCSS).
Guaranteed to converge but may reach a local optimum — run multiple times.

**k-means++ initialization:**
Choose first centroid randomly, then each subsequent centroid with probability
proportional to $d(x)^2$ (distance to nearest existing centroid).
Reduces the chance of poor initialization significantly.

**Hierarchical clustering:**
Builds a tree (dendrogram) of clusters.
- **Agglomerative (bottom-up):** start with each point as its own cluster;
  merge the two closest clusters at each step
- **Divisive (top-down):** start with one cluster; recursively split

**Linkage criteria (how to measure distance between clusters):**
- **Complete linkage:** maximum pairwise distance — tight, circular clusters
- **Single linkage:** minimum pairwise distance — chaining (elongated clusters)
- **Average linkage:** mean pairwise distance — compromise
- **Ward:** minimize total within-cluster variance — compact, equal-size clusters (most common for genomics)

---
## Step 3 — Biological Background

**Cancer subtype discovery:**
- RNA-seq expression of ~500 selected genes across 500 tumor samples
- k-means or hierarchical clustering + heatmap = standard exploratory analysis
- TCGA identified 5 PAM50 breast cancer subtypes this way

**Gene co-expression modules:**
- Hierarchical clustering on the gene-gene correlation matrix
- Genes in the same cluster tend to be co-regulated and functionally related
- WGCNA (Weighted Gene Co-expression Network Analysis) is the standard tool

**scRNA-seq cell typing:**
- Leiden/Louvain clustering on the kNN graph (Module 12)
- k-means on PCA/UMAP coordinates also used
- Cluster quality evaluated by silhouette score

**Choosing k — elbow method:**
Plot WCSS (within-cluster sum of squares) vs. k. The "elbow" is where adding
more clusters stops improving much. Often unclear — use silhouette analysis for
a more principled choice.

**Silhouette score:**
$s(i) = (b(i) - a(i)) / \max(a(i), b(i))$
where $a(i)$ = mean distance to own cluster, $b(i)$ = mean distance to nearest
other cluster. Range $[-1, 1]$; higher = better cluster assignment.

---
## Step 4 — Mathematical Explanation

**k-means objective:**
$$\min_{C_1,\ldots,C_k} \sum_{j=1}^k \sum_{\mathbf{x} \in C_j} \|\mathbf{x} - \boldsymbol{\mu}_j\|_2^2$$

**Centroid update** (exact minimizer of WCSS given assignments):
$$\boldsymbol{\mu}_j = \frac{1}{|C_j|} \sum_{\mathbf{x} \in C_j} \mathbf{x}$$

**Convergence:** WCSS decreases monotonically at each step → finite convergence
(finite states). But may converge to a local minimum.

**k-means++ expected bound:** k-means++ gives an expected WCSS of
$O(\log k) \cdot \text{OPT}$ where OPT is the global optimum.

**Silhouette:**
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}, \quad a(i) = \frac{1}{|C_{c(i)}-1|}\sum_{j \in C_{c(i)}, j \neq i} d(i,j)$$

**Ward linkage:** Merge clusters $A$ and $B$ that minimize:
$$\Delta(A,B) = \frac{|A||B|}{|A|+|B|} \|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2$$

In [ ]:
# Step 6 — Python: k-means from scratch + hierarchical via scipy
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans

# ---- k-means from scratch with k-means++ init ----
class KMeansScratch:
    def __init__(self, k=5, max_iter=300, n_init=10, seed=42):
        self.k = k
        self.max_iter = max_iter
        self.n_init = n_init
        self.seed = seed
    
    def _init_plusplus(self, X, rng):
        """k-means++ initialization."""
        n = len(X)
        centroids = [X[rng.integers(0, n)]]
        for _ in range(self.k - 1):
            dists = np.min([np.sum((X - c)**2, axis=1) for c in centroids], axis=0)
            probs = dists / dists.sum()
            next_idx = rng.choice(n, p=probs)
            centroids.append(X[next_idx])
        return np.array(centroids)
    
    def _run_once(self, X, rng):
        centroids = self._init_plusplus(X, rng)
        labels = np.zeros(len(X), dtype=int)
        for _ in range(self.max_iter):
            # Assignment step
            dists = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
            new_labels = np.argmin(dists, axis=1)
            if np.all(new_labels == labels):
                break
            labels = new_labels
            # Update step
            for j in range(self.k):
                if np.sum(labels == j) > 0:
                    centroids[j] = X[labels == j].mean(axis=0)
        wcss = sum(np.sum((X[labels==j] - centroids[j])**2) for j in range(self.k))
        return labels, centroids, wcss
    
    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        best_wcss = np.inf
        for init_run in range(self.n_init):
            labels, centroids, wcss = self._run_once(X, rng)
            if wcss < best_wcss:
                best_wcss = wcss
                self.labels_ = labels
                self.cluster_centers_ = centroids
                self.inertia_ = wcss
        return self
    
    def predict(self, X):
        dists = np.linalg.norm(X[:, np.newaxis, :] - self.cluster_centers_[np.newaxis, :, :], axis=2)
        return np.argmin(dists, axis=1)

# ---- Generate cancer subtype data ----
rng = np.random.default_rng(42)
n_per_subtype = 100
n_subtypes = 5
n_genes = 50

means = np.zeros((n_subtypes, n_genes))
for i in range(n_subtypes):
    means[i, i*10:(i+1)*10] = 3.0
    means[i, (i*10+5)%n_genes:(i*10+8)%n_genes] = -2.0

X_list = [rng.normal(means[i], 0.8, (n_per_subtype, n_genes)) for i in range(n_subtypes)]
X_cancer = np.vstack(X_list)
y_true = np.repeat(np.arange(n_subtypes), n_per_subtype)
print(f'Cancer data: {len(X_cancer)} samples, {n_genes} genes, {n_subtypes} subtypes')

# ---- Elbow curve ----
wcss_vals = []
silhouette_vals = []
k_range = range(2, 12)
for k in k_range:
    km = KMeansScratch(k=k, n_init=5)
    km.fit(X_cancer)
    wcss_vals.append(km.inertia_)
    sil = silhouette_score(X_cancer, km.labels_)
    silhouette_vals.append(sil)

print('\nSilhouette scores by k:')
for k, sil in zip(k_range, silhouette_vals):
    print(f'  k={k}: {sil:.4f}')
best_k = list(k_range)[np.argmax(silhouette_vals)]
print(f'Best k by silhouette: {best_k}')

# ---- Final clustering with best k ----
km_best = KMeansScratch(k=best_k, n_init=10)
km_best.fit(X_cancer)
ari = adjusted_rand_score(y_true, km_best.labels_)
print(f'\nARI (k-means scratch vs. true labels): {ari:.4f}')

# Compare to sklearn
km_sk = KMeans(n_clusters=best_k, n_init=10, random_state=42)
km_sk.fit(X_cancer)
ari_sk = adjusted_rand_score(y_true, km_sk.labels_)
print(f'ARI (sklearn k-means): {ari_sk:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: Elbow + silhouette
ax = axes[0]
ax2 = ax.twinx()
ax.plot(k_range, wcss_vals, 'steelblue', marker='o', lw=2, ms=5, label='WCSS')
ax2.plot(k_range, silhouette_vals, 'tomato', marker='s', lw=2, ms=5, label='Silhouette')
ax.axvline(best_k, color='grey', ls='--', lw=0.8)
ax.set_xlabel('k')
ax.set_ylabel('WCSS', color='steelblue')
ax2.set_ylabel('Silhouette score', color='tomato')
ax.set_title(f'A. Elbow + Silhouette\n(best k={best_k})')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper right')

# Panel B: PCA visualization colored by k-means labels
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_cancer)
cmap = plt.cm.get_cmap('tab10', n_subtypes)
ax = axes[1]
for subtype in range(best_k):
    mask = km_best.labels_ == subtype
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[cmap(subtype)], s=10, alpha=0.7)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'B. k-means (k={best_k}) on PCA\nARI={ari:.3f}')

# Panel C: Hierarchical clustering dendrogram (subset of 50 samples)
ax = axes[2]
sample_idx = rng.choice(len(X_cancer), 50, replace=False)
X_sub = X_cancer[sample_idx]
y_sub = y_true[sample_idx]
Z = linkage(X_sub, method='ward')
cmap_5 = ['tomato', 'steelblue', 'green', 'orange', 'purple']
label_colors = [cmap_5[y_sub[i]] for i in range(len(y_sub))]
dend = dendrogram(Z, ax=ax, labels=y_sub, leaf_font_size=6,
                    leaf_rotation=90, color_threshold=0.7*Z.max())
ax.set_title('C. Ward hierarchical clustering\n(50-sample subset, label=true subtype)')
ax.set_ylabel('Distance')

# Panel D: Heatmap (samples × genes, sorted by cluster)
ax = axes[3]
sorting = np.argsort(km_best.labels_)
heatmap_data = X_cancer[sorting, :20]  # first 20 genes
im = ax.imshow(heatmap_data.T, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
plt.colorbar(im, ax=ax, shrink=0.5, label='Expression')
# Subtype boundary lines
boundaries = [0]
for st in range(n_subtypes):
    count = np.sum(km_best.labels_ == st)
    boundaries.append(boundaries[-1] + count)
for b in boundaries[1:-1]:
    ax.axvline(b - 0.5, color='white', lw=0.8)
ax.set_xlabel('Samples (sorted by k-means cluster)')
ax.set_ylabel('Gene')
ax.set_title('D. Expression heatmap\n(sorted by cluster assignment)')

plt.suptitle('Module 13 NB10: Clustering — k-Means and Hierarchical', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('clustering.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Run k-means with random initialization (not k-means++) 10 times. Show the
   distribution of WCSS values. Repeat with k-means++. Compare the variance.
2. Apply hierarchical clustering with all four linkage criteria (single, complete,
   average, Ward). Which produces the best ARI on this dataset?
3. Implement the silhouette score from scratch. Verify it matches sklearn.
4. Explain why k-means fails on non-spherical clusters. Show an example with
   two crescent-shaped clusters.

---
## Step 10 — Quiz

1. Describe Lloyd's k-means algorithm in 4 steps.
2. What objective does k-means minimize? Is the solution guaranteed to be global?
3. What is k-means++ and why is it better than random initialization?
4. What does the silhouette score measure?
5. What is Ward linkage and why is it commonly used for biological data?

---
## Step 12 — Reflection

> *[In one paragraph: explain why k-means assumes spherical clusters of similar
> size, and describe a biological scenario where this assumption would fail and
> what method you would use instead.]*

---
*Next: `11_dimensionality_reduction_pca_and_umap.ipynb`*